In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS retailanalytics.config;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS retailanalytics.audit;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS retailanalytics.reject;

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS retailanalytics.watermark;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailanalytics.config.table_config
(
    TableName STRING,
    SourceSystem STRING,
    SourceFile STRING,
    SourceFormat STRING,
    LoadType STRING,
    SCDType STRING,
    ActiveFlag STRING,
    BusinessKey STRING
)
USING DELTA;

In [0]:
config_data = [
("products","github","products.csv","csv","FULL","NA","Y","ProductID"),
("customers","adls","customers.csv","csv","INCREMENTAL","SCD2","Y","CustomerID"),
("orders","adls","orders.csv","csv","INCREMENTAL","NA","Y","OrderID"),
("exchange_rates","api","exchange_rates.json","json","FULL","NA","Y","currency_code")
]

config_df = spark.createDataFrame(config_data,["TableName","SourceSystem","SourceFile","SourceFormat","LoadType","SCDType","ActiveFlag","BusinessKey"])

(config_df.write.format("delta").mode("overwrite").saveAsTable("retailanalytics.config.table_config"))

In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailanalytics.watermark.watermark_control
(
    TableName STRING,
    WatermarkColumn STRING,
    LastWatermarkValue TIMESTAMP,
    UpdatedTimestamp TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.types import *

watermark_schema = StructType([
    StructField("TableName",StringType(),True),
    StructField("WatermarkColumn",StringType(),True),
    StructField("LastWatermarkValue",TimestampType(),True),
    StructField("UpdatedTimestamp",TimestampType(),True)
])
watermark_data = [
    ("customers","LastUpdated",None,None),
    ("orders","OrderDate",None,None)
]
watermark_df = spark.createDataFrame(watermark_data,watermark_schema)

(watermark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
.saveAsTable("retailanalytics.watermark.watermark_control"))

print("Watermark Table Initialized")

Watermark Table Initialized


In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailanalytics.audit.pipeline_audit
(
    PipelineRunId STRING,
    TableName STRING,
    SourceCount BIGINT,
    TargetCount BIGINT,
    RejectCount BIGINT,
    LoadTimestamp TIMESTAMP,
    Status STRING
)
USING DELTA;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailanalytics.reject.reject_records
(
    TableName STRING,
    RejectReason STRING,
    RecordData STRING,
    RejectTimestamp TIMESTAMP
)
USING DELTA;

In [0]:
display(spark.table("retailanalytics.config.table_config"))
display(spark.table("retailanalytics.watermark.watermark_control"))

TableName,SourceFile,SourceFormat,LoadType,SCDType,ActiveFlag
products,products.csv,csv,FULL,NA,Y
customers,customers.csv,csv,INCREMENTAL,SCD2,Y
orders,orders.csv,csv,INCREMENTAL,NA,Y
exchange_rates,exchange_rates.json,json,FULL,NA,Y


TableName,WatermarkColumn,LastWatermarkValue,UpdatedTimestamp
customers,LastUpdated,null,null
orders,OrderDate,null,null
